# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen3-32B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

In [38]:
!git clone https://github.com/Kodro23/Telco-challenge

fatal: destination path 'Telco-challenge' already exists and is not an empty directory.


### 1. Load libraries

In [39]:
import pandas as pd
import numpy as np
import re

### 2.Load data

In [40]:
#load data
df = pd.read_csv("/content/Telco-challenge/train.csv")
print(df.head())

              ID                                           question answer
0  ID_1P7PJMPV0R  Analyze the 5G wireless network drive-test use...     C2
1  ID_8B1D1TUTFA  Analyze the 5G wireless network drive-test use...     C1
2  ID_IGGXMA9GZH  Analyze the 5G wireless network drive-test use...     C2
3  ID_D6C9N2X295  Analyze the 5G wireless network drive-test use...     C2
4  ID_8JC15PNP3Q  Analyze the 5G wireless network drive-test use...     C5


In [41]:
#Function to clean questions
def clean_question(question):
    lines=question.split("\n")
    content=[]
    for line in lines:
        if "|" in line:
            content.append(line.split("|"))
    drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
    engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
    #clean question
    ##recompute begining of the prompt
    q=""
    for l in [l+"\n" for l in lines[:19]]:
        q=q+l
    ##assemble
    cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
    return cleaned_question
#Apply to dataset
df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
print(df["cleaned_question"][0])

 Question: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values repre

## II/ Untrained model

In [42]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
torch.cuda.empty_cache()

In [43]:

# load the tokenizer and the model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [44]:
def find_the_root_cause(question):
  messages = [
    {"role": "user", "content": question}
  ]
  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
    )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  # conduct text completion
  with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128
        )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
  # parsing thinking content
  try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
    index = 0
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

  match = re.search(r"\\boxed\{(C\d)\}", content)
  if match:
      return match.group(1)
  else:
      return "UNKNOWN"


In [45]:
df["root_cause"]=df["cleaned_question"].apply(lambda x: find_the_root_cause(x))

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.29 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.70 GiB is free. Including non-PyTorch memory, this process has 12.86 GiB memory in use. Of the allocated memory 11.75 GiB is allocated by PyTorch, and 1008.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)